# Daily Challenge: Pinecone Serverless Reranking
This notebook demonstrates document reranking, serverless index setup, semantic search, and reranking for medical notes using Pinecone and Hugging Face Transformers.

In [ ]:
# Part 1: Load Documents & Execute Reranking Model
!pip install -U pinecone==6.0.1 pinecone-notebooks

In [ ]:
import os
if not os.environ.get('PINECONE_API_KEY'):
    from pinecone_notebooks.colab import Authenticate
    Authenticate()

In [ ]:
from pinecone import Pinecone
api_key = os.environ.get('PINECONE_API_KEY')
pc = Pinecone(api_key=api_key)

In [ ]:
query = 'Tell me about Apples products'
documents = [
    'Apple is a popular fruit, known for its sweet taste and health benefits.',
    'Apple Inc. designs and sells iPhones, iPads, and Mac computers.',
    'Green apples are tart and often used in baking pies.',
    'Apples latest product launch included new MacBooks and AirPods.',
    'Eating apples daily can improve digestion and overall health.'
]

In [ ]:
from pinecone import RerankModel
reranked = pc.inference.rerank(
    model='bge-reranker-v2-m3',
    query=query,
    documents=[{'id': str(i), 'text': doc} for i, doc in enumerate(documents)],
    top_n=3
)

In [ ]:
def show_reranked_results(query, matches):
    print(f'Query: {query}')
    for i, m in enumerate(matches):
        print(f'{i+1}. Score: {m.score:.4f} | Document: {m.document.text}')
show_reranked_results(query, reranked.matches)

In [ ]:
# Part 2: Setup a Serverless Index for Medical Notes
!pip install pandas torch transformers

In [ ]:
import os
import time
import pandas as pd
from pinecone import Pinecone, ServerlessSpec
from transformers import AutoTokenizer, AutoModel
import torch
cloud = os.getenv('PINECONE_CLOUD', 'aws')
region = os.getenv('PINECONE_REGION', 'us-east-1')
spec = ServerlessSpec(cloud=cloud, region=region)
index_name = 'medical-notes-index'

In [ ]:
if pc.has_index(name=index_name):
    pc.delete_index(name=index_name)
pc.create_index(
    name=index_name,
    dimension=384,
    metric='cosine',
    spec=spec
)

In [ ]:
import requests
import tempfile
with tempfile.TemporaryDirectory() as tmpdirname:
    file_path = os.path.join(tmpdirname, 'sample_notes_data.jsonl')
    url = 'https://raw.githubusercontent.com/pinecone-io/examples/refs/heads/master/docs/data/sample_notes_data.jsonl'
    response = requests.get(url)
    response.raise_for_status()
    with open(file_path, 'wb') as f:
        f.write(response.content)
    df = pd.read_json(file_path, orient='records', lines=True)

In [ ]:
print('Data shape:', df.shape)
df.head()

In [ ]:
index = pc.Index(name=index_name)
index.upsert_from_dataframe(df)

In [ ]:
def is_fresh(index):
    stats = index.describe_index_stats()
    vector_count = stats.total_vector_count
    print('Vector count:', vector_count)
    return vector_count > 0
while not is_fresh(index):
    time.sleep(5)
print('Index ready!')
index.describe_index_stats()

In [ ]:
def get_embedding(input_question):
    model_name = 'sentence-transformers/all-MiniLM-L6-v2'
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    encoded_input = tokenizer(input_question, padding=True, truncation=True, return_tensors='pt')
    with torch.no_grad():
        model_output = model(**encoded_input)
        embedding = model_output.last_hidden_state[0].mean(dim=0) # Average over sequence length
    return embedding

In [ ]:
question = 'patient has chest pain'
query = get_embedding(question).tolist()
results = index.query(vector=[query], top_k=5, include_metadata=True)
sorted_matches = sorted(results['matches'], key=lambda x: x['score'], reverse=True)

In [ ]:
def show_results(question, matches):
    print(f'Question: {question}')
    print('
Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match[id]}')
        print(f' Score: {match[score]}')
        print(f' Metadata: {match[metadata]}')
        print('')
show_results(question, sorted_matches)

In [ ]:
transformed_documents = [
    {
        'id': match['id'],
        'reranking_field': '; '.join([f'{key}: {value}' for key, value in match['metadata'].items()])
    }
    for match in results['matches']
]

In [ ]:
refined_query = 'patient needs knee surgery'
reranked_results = pc.inference.rerank(
    model='bge-reranker-v2-m3',
    query=refined_query,
    documents=transformed_documents,
    rank_fields=['reranking_field'],
    top_n=2,
    return_documents=True
)

In [ ]:
def show_reranked_results(question, matches):
    print(f'Question: {question}')
    print('
Reranked Results:')
    for i, match in enumerate(matches):
        print(f'{str(i+1).rjust(4)}. ID: {match.document.id}')
        print(f' Score: {match.score}')
        print(f' Reranking Field: {match.document.reranking_field}')
        print('')
show_reranked_results(refined_query, reranked_results.matches)

In [ ]:
# Clean up (optional)
pc.delete_index(name=index_name)